# Fine-Tuning SPLADE on Amazon ESCI — Self-Study Notebook

> **Reference material, not meant to run in the workshop.** This notebook is the companion to `notebooks/lab.ipynb`: the lab shows how the finished model behaves in Qdrant, and this notebook shows how a model like that is trained. Code is illustrative — paths, model names, and hyperparameters reflect the real training run, but you'll need a GPU and the full ESCI dataset to actually execute.

## What you'll see

1. Why fine-tune SPLADE on ESCI at all
2. Data prep — loading ESCI, formatting query-product pairs, train/val/test partition
3. Hard-negative mining (ANCE-style)
4. Training loop — `sentence-transformers` + `SparseEncoderTrainer`, hyperparams, ~6 minutes on a single A100 for the 100K-sample fine-tune (article series figure)
5. Evaluation on the held-out ESCI test split with the workshop's `eval.metrics` module
6. Catastrophic forgetting note (the article-4 finding)
7. How to apply the same loop to your own catalog

Workflow connection: run the lab first to see the eval loop and failure analysis; use this notebook next when you want to replace the public ESCI model with one trained on your own labels.

Source: Qdrant's `sparse-embeddings-ecommerce` 5-part article series.

## 1. Why fine-tune SPLADE on ESCI

A SPLADE encoder trained on general web text (e.g. MS MARCO) is competent at passage retrieval, but e-commerce product search has a very different distribution:

- **Short, attribute-dense queries** ("red dress size medium")
- **Brand and model number sensitivity** ("iPhone 15 Pro Max 256GB")
- **Substitute vs Exact distinction** that doesn't exist in web Q&A (256GB vs 128GB are both *about iPhones* but only one is *the iPhone you asked for*)

Fine-tuning teaches SPLADE which terms to expand to (and *not* expand to) in this domain. The result, on the full 2K ESCI test set: a clean lift in nDCG@10 over both BM25 and baseline dense (see the wrap table in `notebooks/lab.ipynb`).

## 2. Data prep

ESCI ships as three files: a query-product pairs file with graded relevance (E/S/C/I), a product catalog, and a train/test split column. The dataset is multilingual; we filter to `us` locale for this run.

In [ ]:
# Illustrative only — requires the ESCI dataset locally
import pandas as pd

examples = pd.read_parquet("esci/shopping_queries_dataset_examples.parquet")
products = pd.read_parquet("esci/shopping_queries_dataset_products.parquet")

examples = examples[examples["product_locale"] == "us"]
df = examples.merge(products, on=["product_id", "product_locale"], how="left")

# ESCI grades -> training labels: Exact=1.0, Substitute=0.1, Complement=0.01, Irrelevant=0.0
GRADE_TO_LABEL = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}
df["label"] = df["esci_label"].map(GRADE_TO_LABEL)

train = df[df["split"] == "train"]
test = df[df["split"] == "test"]

# Reserve ~5% of train as validation, by query_id (no leakage)
val_qids = train["query_id"].drop_duplicates().sample(frac=0.05, random_state=42)
val = train[train["query_id"].isin(val_qids)]
train = train[~train["query_id"].isin(val_qids)]

print(f"train: {len(train):,} pairs · val: {len(val):,} · test: {len(test):,}")

### Step 2: mine hard negatives

Hard negatives are "wrong but plausible" products — labeled S/C/I in ESCI but semantically close to the query. We mine them using a **base dense encoder** (DistilBERT) to find products that look-alike to the query but aren't graded E. This is the ANCE-style pattern — dense for mining, sparse (SPLADE) for the final ranker.

Why dense for mining? Two reasons:

1. A general-purpose dense encoder is already trained on broad semantic similarity — it surfaces "looks like the query" candidates without needing a warmed-up sparse model.
2. Sparse models are trained *against* these mined negatives. Using the same model to mine and to train creates a circular signal that converges slowly.

Procedure:

1. Encode all products with **DistilBERT base (dense)**.
2. For each (query, positive_product) pair, find the top-N nearest products **by cosine similarity** in the dense space.
3. Filter out anything labeled `E` (Exact) for that query — those would be positives.
4. Sample K hard negatives per positive from what's left.
5. Optionally re-mine every epoch — though with a fixed dense miner, once is usually enough.

This is the most important lever in the whole pipeline.


In [ ]:
# Illustrative — pseudocode-ish
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("distilbert-base-uncased")
product_embs = encoder.encode(products["product_title"].tolist(), batch_size=64, show_progress_bar=True)

def mine_hard_negatives(query, positive_pids, k=4, top_n=100):
    q_emb = encoder.encode(query)
    sims = product_embs @ q_emb
    candidate_idx = np.argpartition(-sims, top_n)[:top_n]
    # Drop known positives (any E-graded product for this query)
    candidates = [i for i in candidate_idx if products.iloc[i]["product_id"] not in positive_pids]
    return np.random.choice(candidates, size=k, replace=False)

## 4. Training loop

We use `sentence-transformers` with a `SparseEncoder` head and `SparseMarginMSELoss` against the graded labels, driven by `SparseEncoderTrainer`. The article series reports **~6 minutes on a single A100 for the 100K-sample fine-tune (article series figure), 3 epochs**.

Hyperparameters that mattered:

| Hyperparam | Value | Why |
|---|---|---|
| Learning rate | 2e-5 | Standard for SPLADE fine-tuning; lower destabilizes the sparse regularizer |
| Batch size | 32 (×8 hard negatives) | Bounded by A100 memory with full BERT activations |
| Epochs | 3 | Diminishing returns past 3 on validation nDCG@10 |
| FLOPS regularizer λ_q | 5e-3 | Keeps query vectors sparse (~30 active terms) |
| FLOPS regularizer λ_d | 1e-3 | Keeps document vectors sparse (~150 active terms) |
| Warmup | 10% of total steps | Standard |

In [ ]:
# Illustrative training loop sketch
from sentence_transformers import SparseEncoder, SparseEncoderTrainingArguments, SparseEncoderTrainer
from sentence_transformers.sparse_encoder.losses import SparseMarginMSELoss

model = SparseEncoder("distilbert-base-uncased")
loss = SparseMarginMSELoss(model, lambda_q=5e-3, lambda_d=1e-3)

args = SparseEncoderTrainingArguments(
    output_dir="out/splade-ecommerce-esci",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy="epoch",
    logging_steps=50,
)

trainer = SparseEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_with_hard_negatives,
    eval_dataset=val_with_hard_negatives,
    loss=loss,
)
trainer.train()
model.save_pretrained("out/splade-ecommerce-esci/final")

## 5. Evaluation on the held-out ESCI test split

We use the same `eval.metrics` module the workshop ships with. nDCG@10 is the headline; Recall@10 is the secondary. Keep the evaluation table simple enough that you can read the result quickly and then inspect the queries that moved.

In [ ]:
from eval.metrics import ndcg_at_k, recall_at_k

per_query_ndcg = []
per_query_recall = []

for qid, group in test.groupby("query_id"):
    query_text = group.iloc[0]["query"]
    qrels = dict(zip(group["product_id"], group["esci_label"]))
    retrieved = retrieve_top_k(model, query_text, k=10)  # your retrieval fn
    per_query_ndcg.append(ndcg_at_k(retrieved, qrels, k=10))
    per_query_recall.append(recall_at_k(retrieved, qrels, k=10))

mean_ndcg = sum(per_query_ndcg) / len(per_query_ndcg)
mean_rec = sum(per_query_recall) / len(per_query_recall)

print(f"nDCG@10:   {mean_ndcg:.3f}")
print(f"Recall@10: {mean_rec:.3f}")

## 6. Diagnostic — inspect the active terms

The most useful sanity check after fine-tuning is to look at *which terms the model now activates* on a domain query.

Run a few queries through `eval.viewer.inspect_sparse_vector` and confirm that:

- Attribute tokens that matter (`256gb`, `medium`, `red`, brand names) stay active with non-trivial weight.
- Generic expansions that don't help in e-commerce (`device`, `mobile`, `cellular`, `smartphone`) are pruned.

Sparse vectors are interpretable; this is the cheapest debugging signal you have during a fine-tune.

## 7. Catastrophic forgetting — the article-4 finding

**Important caveat.** Fine-tuning SPLADE on ESCI does *exactly what we asked it to* — and that comes with a cost. The fine-tuned model can perform **worse than a general-domain SPLADE on general-domain retrieval benchmarks** (MS MARCO, BEIR). The workshop wrap showed a related production lesson: non-shopping queries should be routed away from the product index before retrieval.

This is **catastrophic forgetting** — the well-known phenomenon where fine-tuning on a narrow distribution overwrites generalist capability. If you serve a single domain (your product catalog), this is fine, even desirable. If you serve mixed traffic (product search + customer-support Q&A + content discovery), you need either:

- **Router + per-domain models** — classify the query, pick the matching retriever
- **Multi-domain joint fine-tuning** — train on ESCI + MS MARCO + your support corpus together
- **LoRA / adapter fine-tuning** — keep the base weights frozen, swap adapters per domain

## Applying this to your own data

The main lab ends with this loop: baseline → inspect failures → fine-tune → evaluate → hybridize → rerank. This section expands the fine-tune step into a template for your own product catalog and your own labeled queries. Treat it as a 30-day plan, not a runnable cell — the recipe is here; the data is on you.

The rest of this notebook (data prep, hard-negative mining, training, eval) was a worked example on a public dataset. What follows translates each of those stages into "what you need to do" against your catalog and your traffic. Everything that mattered for the ESCI run — graded labels, a meaningful hard-negative set, a leak-free held-out split — matters at least as much on your data, where you don't have the comfort of a curated public benchmark to fall back on.

### What data you need

The training signal SPLADE needs is the same shape no matter the domain — a list (or iterable) of `(query, product_id, grade)` triples:

- **`query`** — the raw user query string, exactly as a shopper typed it
- **`product_id`** — your catalog's SKU or internal identifier
- **`grade`** — ESCI-style 4-tier letters (`E`/`S`/`C`/`I`) or binary 1/0 — both work; the training loss handles either. If you only have binary labels, you'll lose the Substitute/Complement granularity but the model still learns the Exact-vs-Irrelevant boundary, which is the most important one.

**Minimum useful dataset size: ~1K query-product pairs spanning 100–500 unique queries.** The ESCI run used 100K, but that's not the floor — it's the ceiling of "nice to have." Below ~1K pairs the model usually can't learn enough domain signal to beat a well-tuned BM25, so labeling effort is better spent than training effort.

**Where to get labels**, in increasing order of effort and quality:

- **Click logs as weak positive signal.** A clicked SKU after a query is a likely-relevant pair, but clicks are noisy — apply a confidence threshold like ≥3 clicks across distinct sessions before treating one as a positive.
- **Conversion logs as strong positive.** Add-to-cart or purchase after a query is a much cleaner relevance signal; the volume is lower but the precision is much higher.
- **Manual annotation.** Roughly $2–5K via Scale AI / Surge / mTurk for 1K queries × 10 products graded. Slow to set up but the cleanest label source.
- **DIY.** A single annotator can label 200–500 examples in a weekend — useful for sanity-checking your pipeline and your label schema before scaling up.

In practice the strongest setup mixes them: click logs for breadth, conversions for the cleanest positives, manual annotation for the held-out eval set so you can trust the metric.

In [ ]:
# Expected shape — replace with your data loader.
# Each row: one (query, product_id, grade) triple.
your_pairs = [
    {"query": "iPhone 15 Pro Max 256GB",
     "product_id": "B0CHX1Y8Z9",
     "grade": "E"},     # Exact match
    {"query": "iPhone 15 Pro Max 256GB",
     "product_id": "B0CHX1QQQQ",
     "grade": "S"},     # Substitute (e.g. 128GB variant)
    # ... thousands more, ideally 1K+ pairs spanning 100-500 unique queries.
]

# Tip: store as JSONL on disk and load lazily. ~50 bytes per row, so
# 100K pairs is ~5 MB — small enough that pandas/datasets both work fine.

### Mine hard negatives for YOUR data

Hard negatives are the single highest-leverage piece of the whole pipeline. The model learns "iPhone ≠ banana" trivially from any random negative — that signal is essentially free and worth almost nothing. The real lift comes from teaching it "iPhone 15 Pro Max 256GB ≠ iPhone 15 Pro Max 128GB" — products that look almost right but aren't what the shopper asked for. That's the distinction that drives revenue, and you only get it by mining negatives the model actually finds confusing.

The recipe applied to your data:

- **Embed your products** (titles + descriptions, concatenated) with a base dense encoder — DistilBERT or MiniLM via FastEmbed both work. Dense, not sparse, because a general-purpose dense encoder is already trained on broad semantic similarity and will surface "looks like the query" candidates without needing a warmed-up SPLADE.
- **For each training query, retrieve the top-50 nearest products** by cosine similarity in that dense space.
- **Filter out the known-relevant ones** — anything graded `E` or `S` for that query in your training data. Those are positives, not negatives.
- **Take the next 8–16 as hard negatives.** These are the "wrong but plausibly close" products that will actually move the model.

The hard-negative-mining cell earlier in this notebook (the ANCE-style block right after the data-prep section) is the code pattern to copy. Swap the products dataframe for your own, keep the structure.

In [ ]:
# YOUR CONFIG — change these for your data + your run.
TRAIN_PAIRS_PATH        = "your_data/train_pairs.jsonl"
HARD_NEGATIVES_PATH     = "your_data/hard_negatives.jsonl"
OUTPUT_DIR              = "out/your-domain-splade"
BASE_MODEL              = "distilbert-base-uncased"
NUM_EPOCHS              = 3
BATCH_SIZE              = 16
LEARNING_RATE           = 2e-5
HARD_NEGATIVES_PER_QUERY = 8

# The training loop is identical to the cell above — load your pairs +
# hard negatives, wrap in SparseEncoder + SparseEncoderTrainer, fit.
# Expect ~6 min on a single A100 for 100K pairs; longer if your dataset
# is bigger or you're on a smaller GPU.

### How do you know it worked?

The eval recipe is short but every line matters:

- **Hold out 100–200 queries from your training set BEFORE training** — and never train on what you eval on. Data leakage is the #1 silent failure mode in fine-tuning: your metrics look spectacular offline and then production performance is indistinguishable from the baseline. Split by `query_id`, not by row, so that no query appears in both sides of the split.
- **Use the workshop's `eval.metrics.ndcg_at_k` and `recall_at_k`** — the same helpers used in the lab — against the held-out set. nDCG@10 is the headline; Recall@10 is the secondary.
- **Compare to BOTH baselines.** Run BM25 (lexical) and a dense baseline (MiniLM via FastEmbed) on the same held-out queries. If you're only beating BM25, you might just need a sparse model — not necessarily a fine-tune. If you're only beating MiniLM, you might just need a lexical signal. The interesting question is whether the fine-tune lifts you above both.
- **Meaningful lift over BOTH baselines is the bar for "fine-tuning worked."** Look at the aggregate score, then inspect the query-level wins and losses. If the average moves but the examples do not make product-search sense, do not trust the number yet.

### Common pitfalls

Every team that fine-tunes a retriever hits at least two of these. Knowing them in advance saves a sprint.

- **Data leakage** — train set contains test queries → metrics look amazing but production fails.
- **Too few hard negatives** — model learns trivial separations but not the variant-level distinctions (256GB vs 128GB) that drive revenue.
- **Domain narrowness** — fine-tuned only on US English / your specific catalog; degrades when you launch a new geography or a new category.
- **Mixed-intent traffic** — product retrieval should only receive product-search intent; plan a routing layer upstream so non-shopping intents never hit the product index.
- **Catalog drift** — products turn over fast in e-commerce; plan a retrain cadence (quarterly is a reasonable starting point) so the model doesn't go stale against the catalog it's serving.
- **Too small a dataset** — below ~1K pairs the model probably doesn't learn enough domain signal; spend the time labeling more before training.

### What's next

The model is the easy part. Everything around it is where production lift actually comes from.

- **Production deployment.** Query routing (shopping vs general intent), monitoring (drift detection, latency p99), and a rollback plan are most of the work. Budget for them up front, not after the model is "done."
- **A/B test behind a feature flag** before any rollout. Offline metrics correlate with online metrics but don't predict them; verify with real traffic on a small slice before scaling up.
- **Retrain cadence.** Your catalog drifts; your queries drift. Quarterly retrains are a reasonable starting point — and A/B test each new model against the prior so you can catch a regression before it ships to everyone.
- **Optional: cross-encoder rerank on top.** For an additional lift, layer a small cross-encoder over the top-50 retrieved by your fine-tuned SPLADE. The workshop's wrap mentions this as the next layer in the production stack — typically a clean win for ~30ms of added p50 latency.

## 8. Where to go next

- **Multi-domain training.** Mix ESCI with MS MARCO and any internal corpora at the batch level — a uniform per-batch sample from each. Article 4 of the series walks through this.
- **Cross-encoder rerank.** Stack a `ms-marco-MiniLM` or `bge-reranker` on top of the hybrid retrieval. Big quality win for ~30ms of added p50.
- **Active learning loop.** Mine your real production click data for new hard negatives every week. The fine-tuning cost is low enough to re-tune on a cadence.
- **Per-query-type routers.** ESCI labels suggest a natural split between *attribute-dense* and *intent-vague* queries. Different retrievers may win each.

### Resources

- Article series: https://qdrant.tech/articles/sparse-embeddings-ecommerce/
- Released model: https://huggingface.co/thierrydamiba/splade-ecommerce-esci
- ESCI dataset: https://github.com/amazon-science/esci-data
- SPLADE paper: https://arxiv.org/abs/2107.05720
- ANCE paper: https://arxiv.org/abs/2007.00808